# 01 — Synthetic Healthcare Data Generation

This notebook builds the data used by the PoC.

The objective is to create realistic aggregate clinic data without using protected health information. The generated data includes clinic metadata, daily demand, marketing activity, seasonality, weekday effects and capacity pressure.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 80)
plt.rcParams["figure.figsize"] = (11, 4)


In [ ]:
from clinic_forecast.data import SyntheticDataConfig, generate_synthetic_healthcare_data

config = SyntheticDataConfig(
    start_date="2022-01-01",
    end_date="2025-12-31",
    n_clinics=12,
    random_seed=42,
)

usage, metadata, marketing = generate_synthetic_healthcare_data(config)
print("usage", usage.shape)
print("metadata", metadata.shape)
print("marketing", marketing.shape)
usage.head()


## Clinic metadata

Each clinic has a region, specialty, size, daily capacity and baseline staffing level. These variables are useful for both forecasting and decision support.


In [ ]:
metadata


## Demand profile

The target variable is `visits`, defined at clinic-day level. The generator introduces weekly seasonality, yearly seasonality, winter pressure, summer reduction, marketing effects and noise.


In [ ]:
network_daily = usage.groupby("date", as_index=False)["visits"].sum()

fig, ax = plt.subplots()
ax.plot(network_daily["date"], network_daily["visits"])
ax.set_title("Network daily visits")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
plt.show()


In [ ]:
clinic_sample = usage[usage["clinic_id"] == "CLINIC_001"]

fig, ax = plt.subplots()
ax.plot(clinic_sample["date"], clinic_sample["visits"])
ax.set_title("Example clinic daily visits")
ax.set_xlabel("Date")
ax.set_ylabel("Visits")
plt.show()


## Save generated data

The next cell writes the synthetic data into `data/raw`. This makes the later notebooks independent from the generator internals.


In [ ]:
output_dir = PROJECT_ROOT / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

usage.to_csv(output_dir / "clinic_usage.csv", index=False)
metadata.to_csv(output_dir / "clinic_metadata.csv", index=False)
marketing.to_csv(output_dir / "marketing.csv", index=False)

print(f"Saved files to {output_dir}")
